In [1]:
# КЛІТИНКА 1: Імпорти та завантаження
import numpy as np
import pickle
import json
import torch
import random
import pandas as pd
import ast
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from deep_translator import GoogleTranslator
from scipy.special import softmax

with open('knowledge_graph.pkl', 'rb') as f:
    G = pickle.load(f)

with open('graph_metadata.json', 'r', encoding='utf-8') as f:
    metadata = json.load(f)

with open('diagnosis_tests.json', 'r', encoding='utf-8') as f:
    diagnosis_tests = json.load(f)

diagnoses    = metadata['diagnoses']
symptoms     = metadata['symptoms']
procedures   = metadata.get('procedures', [])
adj_matrix   = np.load('adjacency_matrix.npy')
diag_vectors = np.load('diagnosis_vectors.npy')
symp_vectors = np.load('symptom_name_vectors.npy')

df = pd.read_csv('../Dataset/release_train_patients')

print(f'Дiагнозiв: {len(diagnoses)}')
print(f'Симптомiв: {len(symptoms)}')
print(f'Процедур:  {len(procedures)}')
print(f'Пацiєнтiв: {len(df):,}')

Дiагнозiв: 49
Симптомiв: 223
Процедур:  15
Пацiєнтiв: 1,025,602


In [2]:
# КЛІТИНКА 2: BioBERT
MODEL_NAME = 'dmis-lab/biobert-base-cased-v1.2'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
model      = AutoModel.from_pretrained(MODEL_NAME)
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)
print(f'BioBERT. Пристрiй: {device}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BioBERT. Пристрiй: cpu


In [3]:
# КЛІТИНКА 3: Допомiжнi функцiї
_embed_cache: dict = {}


def encode_text(text: str) -> np.ndarray:
    inputs = tokenizer(text, return_tensors='pt',
                       truncation=True, max_length=64,
                       padding=True).to(device)
    with torch.no_grad():
        out = model(**inputs)
    return out.last_hidden_state[:, 0, :].cpu().numpy().squeeze()


def normalize_vec(v: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(v)
    return v / n if n > 0 else v


def normalize_mat(m: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(m, axis=1, keepdims=True)
    norms[norms == 0] = 1
    return m / norms


def translate_to_english(text: str) -> str:
    try:
        return GoogleTranslator(source='auto', target='en').translate(text)
    except Exception:
        return text


def embed_action(action: str) -> np.ndarray:
    if action not in _embed_cache:
        eng = translate_to_english(action)
        _embed_cache[action] = normalize_vec(encode_text(eng))
    return _embed_cache[action]


def compute_probabilities(activations: np.ndarray, tau: float = 1.0) -> np.ndarray:
    return softmax(activations / tau)


def shannon_entropy(probs: np.ndarray) -> float:
    probs = np.clip(probs, 1e-10, 1.0)
    return float(-np.sum(probs * np.log2(probs)))

def get_tau(step: int,
            tau_start: float = 1.2,
            tau_end:   float = 0.35,
            n_steps:   int   = 10) -> float:
    alpha = (tau_start - tau_end) / n_steps
    return max(tau_end, tau_start - alpha * step)


print('Функцiї визначено')
print(f'Temperature schedule: t(0)={get_tau(0):.2f}  '
      f't(5)={get_tau(5):.2f}  t(10)={get_tau(10):.2f}')

Функцiї визначено
Temperature schedule: t(0)=1.20  t(5)=0.78  t(10)=0.35


In [4]:
# КЛІТИНКА 4: VirtualPatient — ВИПРАВЛЕНО
class VirtualPatient:
    _DEMO_WORDS = {
        'age', 'old', 'young', 'year', 'male', 'female',
        'man', 'woman', 'gender', 'sex', 'boy', 'girl',
        'adult', 'child', 'elderly', 'patient'
    }

    def __init__(self,
                 df,
                 diagnoses,
                 symptoms,
                 reveal_ratio:            float = 0.3,
                 similarity_threshold:    float = 0.82,  # Fix 1
                 max_reveal_per_question: int   = 3):    # Fix 2

        self.diagnoses     = diagnoses
        self.symptoms      = symptoms
        self.sim_threshold = similarity_threshold
        self.max_reveal    = max_reveal_per_question

        row = df.sample(1).iloc[0]
        self.true_diagnosis = row['PATHOLOGY']

        # Fix 3: демографiя — окремо
        self.age = (int(row['AGE'])
                    if 'AGE' in row.index and pd.notna(row['AGE']) else None)
        self.sex = (str(row['SEX']).strip()
                    if 'SEX' in row.index and pd.notna(row['SEX']) else None)

        # Симптоми
        evid_list  = ast.literal_eval(row['EVIDENCES'])
        base_codes = list({ev.split('_@_')[0] for ev in evid_list})

        self.patient_symptoms: dict = {}
        for code in base_codes:
            name = code
            for s in symptoms:
                if code in str(s):
                    name = s
                    break
            self.patient_symptoms[name] = False

        all_s    = list(self.patient_symptoms.keys())
        n_reveal = max(1, int(len(all_s) * reveal_ratio))
        for s in random.sample(all_s, n_reveal):
            self.patient_symptoms[s] = True

        self.revealed_symptoms = [s for s, v in self.patient_symptoms.items() if v]
        self.hidden_symptoms   = [s for s, v in self.patient_symptoms.items() if not v]

    # ── Демографiчнi питання ──────────────────────────────────────────
    def _is_demographic(self, question: str) -> bool:
        words = set(question.lower().split())
        return bool(words & self._DEMO_WORDS)

    def _answer_demographic(self, question: str) -> tuple:
        q = question.lower()
        if self.age is not None and any(w in q for w in ('age', 'old', 'year')):
            return f'I am {self.age} years old.', []
        if self.sex is not None and any(
                w in q for w in ('male', 'female', 'man', 'woman',
                                  'gender', 'sex', 'boy', 'girl')):
            sex_str = 'male' if str(self.sex).upper() in ('M', 'MALE') else 'female'
            return f'I am {sex_str}.', []
        return 'I cannot provide that information.', []

    # ── Симптомне питання ─────────────────────────────────────────────
    def answer_question(self, question: str) -> tuple:
        if self._is_demographic(question):
            return self._answer_demographic(question)

        q_vec = normalize_vec(encode_text(translate_to_english(question)))
        newly_revealed = []

        # Крок 1: ПРИХОВАНI симптоми — збираємо кандидатiв, потiм top-N
        candidates = []
        for symp in list(self.hidden_symptoms):
            s_vec = normalize_vec(encode_text(symp))
            sim   = float(cosine_similarity(
                q_vec.reshape(1, -1), s_vec.reshape(1, -1))[0][0])
            if sim >= self.sim_threshold:
                candidates.append((symp, sim))

        # Вiдкриваємо лише top-max_reveal за similarity
        candidates.sort(key=lambda x: x[1], reverse=True)
        for symp, sim in candidates[: self.max_reveal]:
            self.patient_symptoms[symp] = True
            self.revealed_symptoms.append(symp)
            self.hidden_symptoms.remove(symp)
            newly_revealed.append((symp, sim))

        if newly_revealed:
            names = [s for s, _ in newly_revealed]
            return f'Yes: {", ".join(names)}', newly_revealed

        # Крок 2: ВIДКРИТI симптоми — пiдтвердження
        confirmed = []
        for symp in self.revealed_symptoms:
            s_vec = normalize_vec(encode_text(symp))
            sim   = float(cosine_similarity(
                q_vec.reshape(1, -1), s_vec.reshape(1, -1))[0][0])
            if sim >= self.sim_threshold:
                confirmed.append(symp)

        if confirmed:
            return f'Yes, confirmed: {", ".join(confirmed)}', []
        return 'No, patient does not have that symptom.', []

    def get_initial_complaint(self) -> str:
        if self.revealed_symptoms:
            return f'Patient complains of: {self.revealed_symptoms[0]}'
        return 'Patient came in for a check-up'

    def get_status(self) -> dict:
        return {
            'true_diagnosis': self.true_diagnosis,
            'age':            self.age,
            'sex':            self.sex,
            'total_symptoms': len(self.patient_symptoms),
            'revealed':       len(self.revealed_symptoms),
            'hidden':         len(self.hidden_symptoms),
            'revealed_list':  self.revealed_symptoms,
        }


print('VirtualPatient (виправлено) визначено')

VirtualPatient (виправлено) визначено


In [5]:
# КЛІТИНКА 5: PatientModel — ВИПРАВЛЕНО
class PatientModel:
    def __init__(self,
                 diagnoses, symptoms,
                 adj_matrix, diag_vectors, symp_vectors,
                 beta:  float = 0.4,
                 decay: float = 0.85,   # Fix 5: було 0.7
                 top_k: int   = 5):

        self.diagnoses    = diagnoses
        self.symptoms     = symptoms
        self.diag_vectors = normalize_mat(diag_vectors.copy())
        self.symp_vectors = normalize_mat(symp_vectors.copy())
        self.beta         = beta
        self.decay        = decay
        self.top_k        = top_k

        # TF-IDF + L2 нормалiзацiя рядкiв
        N    = len(diagnoses)
        df_  = np.sum(adj_matrix > 0, axis=0)
        df_[df_ == 0] = 1
        idf       = np.log(N / df_)
        adj_tfidf = adj_matrix * idf[np.newaxis, :]
        row_norms = np.linalg.norm(adj_tfidf, axis=1, keepdims=True)
        row_norms[row_norms == 0] = 1
        self.adj_matrix = adj_tfidf / row_norms

        self.diag_scores = np.zeros(len(diagnoses))
        self.symp_scores = np.zeros(len(symptoms))
        self.history: list = []

    # ── Snapshot / Restore для симуляцiй ─────────────────────────────
    def snapshot(self) -> dict:
        """Легка копiя поточного стану (без history)."""
        return {
            'diag': self.diag_scores.copy(),
            'symp': self.symp_scores.copy(),
        }

    def restore(self, snap: dict) -> None:
        """Вiдновлює стан зi знiмку."""
        self.diag_scores = snap['diag'].copy()
        self.symp_scores = snap['symp'].copy()

    def get_diagnosis_activations(self) -> np.ndarray:
        return self.diag_scores

    def update_state(self,
                     action_vector: np.ndarray,
                     revealed_symptoms: list = None) -> np.ndarray:
        """Оновлює стан графа пiсля дiї.
        action_vector     -- L2-нормований BioBERT-вектор питання.
        revealed_symptoms -- iмена пiдтверджених симптомiв (сигнал=1.0).
        """
        self.history.append(self.snapshot())

        symp_sim    = cosine_similarity(
            action_vector.reshape(1, -1), self.symp_vectors)[0]
        sparse_symp = np.zeros_like(symp_sim)
        top_k_idx   = np.argsort(symp_sim)[::-1][: self.top_k]
        sparse_symp[top_k_idx] = symp_sim[top_k_idx]

        if revealed_symptoms:
            for name in revealed_symptoms:
                if name in self.symptoms:
                    idx = self.symptoms.index(name)
                    sparse_symp[idx] = max(sparse_symp[idx], 1.0)

        # Fix 5: decay = 0.85
        self.symp_scores = self.decay * self.symp_scores + self.beta * sparse_symp
        diag_from_symp   = self.adj_matrix @ self.symp_scores
        diag_sim         = cosine_similarity(
            action_vector.reshape(1, -1), self.diag_vectors)[0]

        self.diag_scores = (
            self.decay * self.diag_scores
            + 0.7 * diag_from_symp
            + 0.3 * self.beta * diag_sim
        )
        return np.concatenate([self.diag_scores, self.symp_scores])


print('PatientModel (виправлено) визначено')

PatientModel (виправлено) визначено


In [6]:
# КЛІТИНКА 6: AdaptiveQuestionSelector — НОВИЙ КЛАС
class AdaptiveQuestionSelector:

    CANDIDATE_QUESTIONS = [
        # -- Кардiо / Пульмо -------------------------------------------
        'Do you have chest pain?',
        'Do you have shortness of breath or dyspnea?',
        'Do you have a cough?',
        'Is your cough productive or dry?',
        'Do you cough up blood?',
        'Do you hear wheezing when you breathe?',
        'Do you have rapid or labored breathing?',
        'Are breath sounds decreased on one side?',
        'Do you have palpitations or irregular heartbeat?',
        'Do you have swelling in your legs or ankles?',
        'Did the chest pain radiate to your arm or jaw?',
        'Is the pain worse when breathing deeply?',
        'Did the pain appear suddenly?',
        'Is the pain sharp or dull?',
        'Do you have high blood pressure?',
        # -- Загальнi симптоми -----------------------------------------
        'Do you have fever?',
        'Do you have night sweats?',
        'Do you have fatigue or weakness?',
        'Have you lost weight recently?',
        'Do you have chills?',
        # -- ШКТ --------------------------------------------------------
        'Do you have abdominal pain?',
        'Do you have nausea or vomiting?',
        'Do you have diarrhea?',
        'Do you have blood in your stool?',
        'Do you have jaundice?',
        # -- Неврологiчнi -----------------------------------------------
        'Do you have headache?',
        'Do you have dizziness or loss of balance?',
        'Have you had a seizure?',
        'Do you have tingling or numbness in your limbs?',
        # -- Урологiчнi -------------------------------------------------
        'Do you have pain when urinating?',
        'Do you have frequent urination?',
        # -- Musculoskeletal --------------------------------------------
        'Do you have joint pain or swelling?',
        'Do you have back pain?',
        # -- Анамнез ----------------------------------------------------
        'Have you ever had this condition before?',
        'Are you taking any medications?',
        'Have you recently travelled abroad?',
        'Do you smoke?',
        'Do you consume alcohol regularly?',
        'Do you have any known allergies?',
    ]

    def __init__(self, patient_model: PatientModel, tau_fn):
        self.model   = patient_model
        self.tau_fn  = tau_fn
        self.asked: set = set()

        # Pre-embed всiх кандидатiв одразу
        print('Pre-computing candidate embeddings...')
        self.q_vecs: dict = {
            q: embed_action(q) for q in self.CANDIDATE_QUESTIONS
        }
        print(f'Cached {len(self.q_vecs)} candidate questions')

    def select_best(self, step: int) -> tuple:
        tau      = self.tau_fn(step)
        H_before = shannon_entropy(
            compute_probabilities(self.model.get_diagnosis_activations(), tau))

        best_q:  str   = None
        best_ig: float = -np.inf
        snap = self.model.snapshot()   # один snapshot на весь цикл

        for q, q_vec in self.q_vecs.items():
            if q in self.asked:
                continue

            self.model.update_state(q_vec, revealed_symptoms=None)
            H_after = shannon_entropy(
                compute_probabilities(
                    self.model.get_diagnosis_activations(), tau))
            ig = H_before - H_after
            self.model.restore(snap)   # вiдновлюємо пiсля кожної симуляцiї

            if ig > best_ig:
                best_ig = ig
                best_q  = q

        self.asked.add(best_q)
        return best_q, best_ig

print('AdaptiveQuestionSelector визначено')

AdaptiveQuestionSelector визначено


In [7]:
# КЛІТИНКА 7: Повна симуляцiя з адаптивним вибором питань
random.seed(42)
N_STEPS = 12

virtual_patient = VirtualPatient(
    df=df,
    diagnoses=diagnoses,
    symptoms=symptoms,
    reveal_ratio=0.3,
    similarity_threshold=0.82,
    max_reveal_per_question=3,
)

patient_model = PatientModel(
    diagnoses=diagnoses,
    symptoms=symptoms,
    adj_matrix=adj_matrix,
    diag_vectors=diag_vectors,
    symp_vectors=symp_vectors,
    beta=0.4,
    decay=0.85,
    top_k=5,
)

selector = AdaptiveQuestionSelector(
    patient_model=patient_model,
    tau_fn=get_tau,
)

status_init = virtual_patient.get_status()
tau_0       = get_tau(0)
probs_0     = compute_probabilities(
    patient_model.get_diagnosis_activations(), tau_0)
H_0 = shannon_entropy(probs_0)

print('=' * 65)
print('НОВИЙ ПАЦIЄНТ ЗГЕНЕРОВАНИЙ')
print('=' * 65)
print(f'  Справжнiй дiагноз (прихований): {status_init["true_diagnosis"]}')
print(f'  Вiк: {status_init["age"]}    Стать: {status_init["sex"]}')
print(f'  Всього симптомiв: {status_init["total_symptoms"]}')
print(f'  Вiдкрито одразу:  {status_init["revealed"]}  '
      f'({status_init["revealed_list"]})')
print(f'  Приховано:        {status_init["hidden"]}')
print(f'\nПочаткова скарга: "{virtual_patient.get_initial_complaint()}"')
print(f'H(S_0) = {H_0:.4f}  (tau = {tau_0:.2f})')


def ask_patient(question: str, step: int, H_prev: float) -> float:
    tau = get_tau(step)
    print(f'\nКрок {step}: "{question}"')

    answer, newly_revealed = virtual_patient.answer_question(question)
    print(f'  Пацiєнт: {answer}')

    a_vec          = embed_action(question)
    revealed_names = [s for s, _ in newly_revealed]
    patient_model.update_state(a_vec, revealed_symptoms=revealed_names)

    diag_act  = patient_model.get_diagnosis_activations()
    probs     = compute_probabilities(diag_act, tau=tau)
    H_new     = shannon_entropy(probs)
    dH        = H_prev - H_new

    true_idx  = (diagnoses.index(virtual_patient.true_diagnosis)
                 if virtual_patient.true_diagnosis in diagnoses else -1)
    true_rank = (np.argsort(probs)[::-1].tolist().index(true_idx) + 1
                 if true_idx >= 0 else -1)

    label = 'корисна' if dH > 0 else 'нерелевантна'
    print(f'  H = {H_new:.4f}   dH = {dH:+.4f}  {label}  (tau={tau:.2f})')
    print(f'  Справжнiй дiагноз "{virtual_patient.true_diagnosis}": '
          f'мiсце #{true_rank}  (p={probs[true_idx]:.4f})')
    print('  Топ-3:')
    for i in np.argsort(probs)[::-1][:3]:
        bar    = '█' * int(probs[i] * 40)
        marker = ' <--' if i == true_idx else ''
        print(f'    {probs[i]:.4f} {bar}  {diagnoses[i]}{marker}')

    if newly_revealed:
        cur = virtual_patient.get_status()
        print(f'  Новi симптоми: {revealed_names}')
        print(f'  Вiдкрито: {cur["revealed"]}/{cur["total_symptoms"]}')

    return H_new


print('\n' + '=' * 65)
print('СЕСIЯ ДIАГНОСТИКИ  (адаптивний вибiр питань)')
print('=' * 65)

H_prev = H_0
for step in range(1, N_STEPS + 1):
    best_q, pred_ig = selector.select_best(step)
    print(f'\n  [Selector] Обрано: "{best_q}"  (pred IG = {pred_ig:+.4f})')
    H_prev = ask_patient(best_q, step, H_prev)

print(f'\n{"="*65}')
print('ПIДСУМОК СЕСIЇ:')
status_fin = virtual_patient.get_status()
tau_fin    = get_tau(N_STEPS)
diag_act   = patient_model.get_diagnosis_activations()
probs      = compute_probabilities(diag_act, tau_fin)
top1_idx   = np.argmax(probs)
true_idx   = (diagnoses.index(status_fin['true_diagnosis'])
              if status_fin['true_diagnosis'] in diagnoses else -1)
true_rank  = (np.argsort(probs)[::-1].tolist().index(true_idx) + 1
              if true_idx >= 0 else -1)
correct    = diagnoses[top1_idx] == status_fin['true_diagnosis']

print(f'  Справжнiй дiагноз:    {status_fin["true_diagnosis"]}')
print(f'  Виявлено симптомiв:   '
      f'{status_fin["revealed"]} / {status_fin["total_symptoms"]}')
print(f'  Початкова ентропiя:   {H_0:.4f}')
print(f'  Фiнальна ентропiя:    {H_prev:.4f}')
print(f'  Зниження ентропiї:    {H_0 - H_prev:.4f}')
print(f'  Топ-1 дiагноз:        '
      f'{diagnoses[top1_idx]} (p={probs[top1_idx]:.4f})')
print(f'  Справжнiй дiагноз:    мiсце #{true_rank} (p={probs[true_idx]:.4f})')
print(f'  Результат:            '
      f'{"ПРАВИЛЬНО!" if correct else "Неправильно"}')

Pre-computing candidate embeddings...
Cached 39 candidate questions
НОВИЙ ПАЦIЄНТ ЗГЕНЕРОВАНИЙ
  Справжнiй дiагноз (прихований): Allergic sinusitis
  Вiк: 33    Стать: F
  Всього симптомiв: 7
  Вiдкрито одразу:  2  (['E_226', 'E_207'])
  Приховано:        5

Початкова скарга: "Patient complains of: E_226"
H(S_0) = 5.6147  (tau = 1.20)

СЕСIЯ ДIАГНОСТИКИ  (адаптивний вибiр питань)

  [Selector] Обрано: "Do you have high blood pressure?"  (pred IG = +0.0014)

Крок 1: "Do you have high blood pressure?"
  Пацiєнт: Yes: E_124, E_204, E_181
  H = 5.6133   dH = +0.0014  корисна  (tau=1.11)
  Справжнiй дiагноз "Allergic sinusitis": мiсце #44  (p=0.0199)
  Топ-3:
    0.0233   Stable angina
    0.0230   Possible NSTEMI / STEMI
    0.0228   Unstable angina
  Новi симптоми: ['E_124', 'E_204', 'E_181']
  Вiдкрито: 5/7

  [Selector] Обрано: "Do you consume alcohol regularly?"  (pred IG = +0.0074)

Крок 2: "Do you consume alcohol regularly?"
  Пацiєнт: Yes: E_169, E_86
  H = 5.6056   dH = +0.0077  ко

In [8]:
# КЛІТИНКА 8: Збереження артефактiв

with open('patient_model_class.pkl', 'wb') as f:
    pickle.dump(patient_model, f)

with open('virtual_patient_class.pkl', 'wb') as f:
    pickle.dump(virtual_patient, f)

with open('adaptive_selector_class.pkl', 'wb') as f:
    pickle.dump(selector, f)

scoring_src = '''\
import numpy as np
from scipy.special import softmax


def compute_probabilities(activations, tau=1.0):
    return softmax(activations / tau)


def shannon_entropy(probs):
    probs = np.clip(probs, 1e-10, 1.0)
    return float(-np.sum(probs * np.log2(probs)))


def get_tau(step, tau_start=1.2, tau_end=0.35, n_steps=10):
    """Temperature annealing schedule."""
    alpha = (tau_start - tau_end) / n_steps
    return max(tau_end, tau_start - alpha * step)
'''

with open('scoring_functions.py', 'w', encoding='utf-8') as f:
    f.write(scoring_src)

print('patient_model_class.pkl       OK')
print('virtual_patient_class.pkl     OK')
print('adaptive_selector_class.pkl   OK')
print('scoring_functions.py          OK')

patient_model_class.pkl       OK
virtual_patient_class.pkl     OK
adaptive_selector_class.pkl   OK
scoring_functions.py          OK
